In [1]:
import pandas as pd
comparison = pd.read_excel("/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/ALL_STOCKS_COMPARISON.xlsx")

# then any ad-hoc question, e.g.:
comparison[comparison["Median duration (days)"] > 25][["symbol", "Median duration (days)"]]

,symbol,Median duration (days)
37,BRITANNIA,26.9
54,DELHIVERY,26.2
124,MANKIND,26.8
157,PGEL,26.1
188,TATACONSUM,27.0
207,VMM,27.8
209,WAAREEENER,25.2


In [2]:
import pandas as pd
vbl_flips = pd.read_excel("../exports/VBL_full_report.xlsx", sheet_name="Flips")
vbl_flips[["direction", "status", "duration", "candles_held"]].sort_values("candles_held", ascending=False).head(10)

,direction,status,duration,candles_held
6,UP,CLOSED,150.166667,208
22,UP,CLOSED,123.166667,169
45,DOWN,CLOSED,81.166667,114
51,DOWN,CLOSED,76.166667,102
19,DOWN,CLOSED,63.166667,90
18,UP,CLOSED,65.166667,88
50,UP,CLOSED,55.166667,76
20,UP,CLOSED,53.166667,70
55,DOWN,OPEN,NaN,67
47,DOWN,CLOSED,47.166667,66


In [3]:
index_names = ["NIFTY", "BANKNIFTY", "SENSEX", "NIFTYNXT50", "MIDCPNIFTY", "FINNIFTY"]

comparison_indices = comparison[comparison["symbol"].isin(index_names)].reset_index(drop=True)
comparison_indices

,symbol,Total flips (closed),UP flips,DOWN flips,Avg duration (days),Median duration (days),Longest duration (days),Shortest duration (days),75th percentile duration (days),90th percentile duration (days),...,Smallest monthly range_pct,Largest monthly range_pct,75th percentile monthly range_pct,90th percentile monthly range_pct,Avg monthly up_move_pct,Avg monthly down_move_pct,Largest single up_move_pct,Largest single down_move_pct,75th percentile up_move_pct,75th percentile down_move_pct
0,BANKNIFTY,70,35,35,22.5,19.6,99.0,0.2,30.8,38.3,...,1.56,17.01,9.22,12.34,3.79,3.45,13.45,15.37,5.49,4.49
1,FINNIFTY,80,40,40,19.5,14.2,74.2,0.2,27.4,38.3,...,1.46,15.57,8.16,12.74,3.72,3.40,13.71,14.27,5.20,4.64
2,MIDCPNIFTY,80,40,40,19.3,14.0,83.2,0.2,26.0,41.5,...,1.14,16.90,9.83,12.73,3.99,4.09,12.72,12.40,5.81,6.40
3,NIFTY,76,38,38,20.7,15.2,68.0,0.2,23.7,46.5,...,1.41,12.39,7.25,10.12,3.05,2.98,9.35,9.63,4.08,4.05
4,NIFTYNXT50,82,41,41,19.1,15.1,76.2,0.2,27.2,38.7,...,1.11,18.74,9.99,13.56,3.87,4.26,15.16,14.40,5.11,6.09
5,SENSEX,76,38,38,20.7,15.6,63.2,0.2,25.9,46.5,...,1.18,12.32,7.36,10.18,3.08,2.98,9.00,8.62,4.17,4.17


In [4]:
import sys
sys.path.append("..")

import pandas as pd

# Credit spread shortlist (same logic as 15_daily_screener.ipynb's Cell 1)
credit_spread_ranking = pd.read_excel("../exports/RANKING_credit_spreads.xlsx")
cs_clean = credit_spread_ranking[
    credit_spread_ranking["gap_risk_flag"].isna() | (credit_spread_ranking["gap_risk_flag"] == "")
]
cs_top_25 = cs_clean.head(25)
cs_must_include = credit_spread_ranking[credit_spread_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
cs_must_include_clean = cs_must_include[
    cs_must_include["gap_risk_flag"].isna() | (cs_must_include["gap_risk_flag"] == "")
]
credit_spread_final = pd.concat([cs_top_25, cs_must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)

# Short IC shortlist (same as notebook 07's Cell 1)
short_ic_ranking = pd.read_excel("../exports/RANKING_short_iron_condor.xlsx")
ic_clean = short_ic_ranking[
    (short_ic_ranking["gap_risk_flag"].isna() | (short_ic_ranking["gap_risk_flag"] == "")) &
    (short_ic_ranking["volatility_persistence_flag"].isna() | (short_ic_ranking["volatility_persistence_flag"] == ""))
]
ic_top_25 = ic_clean.head(25)
ic_must_include = short_ic_ranking[short_ic_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
ic_must_include_clean = ic_must_include[
    (ic_must_include["gap_risk_flag"].isna() | (ic_must_include["gap_risk_flag"] == "")) &
    (ic_must_include["volatility_persistence_flag"].isna() | (ic_must_include["volatility_persistence_flag"] == ""))
]
short_ic_final = pd.concat([ic_top_25, ic_must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)

# Side-by-side comparison
comparison_table = pd.DataFrame({
    "credit_spread_pick": credit_spread_final["symbol"].reset_index(drop=True),
    "short_ic_pick": short_ic_final["symbol"].reset_index(drop=True)
})
comparison_table

,credit_spread_pick,short_ic_pick
0,NESTLEIND,NIFTY
1,ITC,SENSEX
2,SUNPHARMA,FINNIFTY
3,DRREDDY,MIDCPNIFTY
4,COLPAL,NIFTYNXT50
5,ULTRACEMCO,BANKNIFTY
6,HAVELLS,SUNPHARMA
7,TECHM,ULTRACEMCO
8,ICICIPRULI,DABUR
9,PIIND,GRASIM


In [5]:
import pandas as pd
summary = pd.read_excel("../exports/BANKNIFTY_full_report.xlsx", sheet_name="Summary")

metrics_needed = [
    "Median monthly range_pct", "75th percentile monthly range_pct", "90th percentile monthly range_pct",
    "Avg monthly up_move_pct", "Avg monthly down_move_pct",
    "75th percentile up_move_pct", "75th percentile down_move_pct",
    "90th percentile up_move_pct", "90th percentile down_move_pct"
]
print(summary[summary["metric"].isin(metrics_needed)])

                               metric  value
25           Median monthly range_pct   6.20
28  75th percentile monthly range_pct   9.22
29  90th percentile monthly range_pct  12.34
30            Avg monthly up_move_pct   3.79
31          Avg monthly down_move_pct   3.45
34        75th percentile up_move_pct   5.49
35      75th percentile down_move_pct   4.49
